# Sinkhorn-normalized diffusion on a point cloud

Quickstart for the numpy core of `sinkhornkernels`, the official implementation of
*Sinkhorn Normalization of Diffusion Kernels* (ICML 2026).

Given points $x_i$ with masses $m_i$ and a Gaussian kernel
$K_{ij} = \exp(-\|x_i - x_j\|^2 / 2\sigma^2)$, the symmetric Sinkhorn
normalization finds the unique positive diagonal $\Lambda$ such that
$Q = \Lambda K M \Lambda$ is a **diffusion operator**: unit row sums, self-adjoint
for the mass-weighted inner product, spectrum in $[0, 1]$.

Note that any other positive definite kernel can be used instead of Gaussian.

This notebook only needs the base install (numpy/scipy) plus matplotlib:

```bash
pip install -e ".[dev]" matplotlib
```

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sinkhornkernels import (
    sinkhorn_log,
    sinkhorn_log_sparse,
    gaussian_diffusion,
    diffusion_eigsh,
    laplacian_eigenvalues,
    squared_distances,
    knn_graph,
    knn_to_sparse_sqdist,
)
from sinkhornkernels import mesh as skmesh

import plot_utils as plu

# pick the 3D rendering backend once: "pyvista" (interactive) or "matplotlib"
plu.set_backend("matplotlib")

rng = np.random.default_rng(0)

## Load a shape and sample a point cloud

We use the Stanford armadillo normalized to a unit-radius sphere,
and sample points uniformly on the surface with uniform weights $m_i = 1/N$.

In [ ]:
vertices, faces = skmesh.load_obj("../data/armadillo.obj")
vertices -= vertices.mean(axis=0)
vertices /= np.linalg.norm(vertices, axis=1).max()

N = 4000
points = skmesh.sample_surface(vertices, faces, N, rng=rng)
masses = np.full(N, 1.0 / N)
sigma = 0.05  # kernel bandwidth, in units of the shape radius

points.shape, masses.sum()

## Sinkhorn convergence

The log-domain solver is stable for **any** bandwidth $\sigma$. In the limit $\sigma\to0$, the result of the normalization is simply the identity matrix.
In practice, convergence is fast. Here, we track the $L^1(\mu)$ marginal violation $\frac{1}{\sum_i m_i}\sum_i m_i |(Q\mathbf{1})_i - 1|$.

Note that a single iteration correspond to standard symmetric normalization used in graph learning.

In [ ]:
sqdist = squared_distances(points)

errors = []
f = None
for it in range(15):
    f, info = sinkhorn_log(sqdist, sigma, masses=masses, n_iter=1, f_init=f, full_output=True)
    errors.append(info["marginal_error"])

plt.figure(figsize=(5, 3.5))
plt.semilogy(np.arange(1, 16), errors, "o-")
plt.xlabel("Sinkhorn iteration")
plt.ylabel("marginal violation (L1)")
plt.title("Convergence of the symmetric Sinkhorn loop")
plt.grid(alpha=0.3)
plt.tight_layout()

## Sparse k-NN solver

For large clouds, the kernel can be truncated to a symmetrized k-NN graph
(the diagonal term is added analytically). Cost per iteration drops from
$O(N^2)$ to $O(kN)$.

In [ ]:
knn_sqdist, knn_idx = knn_graph(points, 64)
sq_sparse = knn_to_sparse_sqdist(knn_sqdist, knn_idx, symmetrize="max")

f_dense = sinkhorn_log(sqdist, sigma, masses=masses, n_iter=20)
f_sparse = sinkhorn_log_sparse(sq_sparse, sigma, masses=masses, n_iter=20)

print(f"stored entries: {sq_sparse.nnz} / {N * N}")
print(f"max |f_sparse - f_dense| = {np.abs(f_sparse - f_dense).max():.2e}")

## The diffusion operator in action

`gaussian_diffusion` creates an abstract Linear Operator for all variatnts of normalization, which can be used as
black box matrix-vector operator `Q`. Applying powers of `Q` to a Dirac diffuses it over
the shape while conserving total mass exactly:

In [ ]:
Q = gaussian_diffusion(points, sigma, masses=masses, mode="sinkhorn", n_iter=20)
print(f"marginal error: {Q.marginal_error():.2e}")

dirac = np.zeros(N)
dirac[0] = 1.0 / masses[0]

panels = []
for n_steps in [1, 5, 25]:
    signal = dirac.copy()
    for _ in range(n_steps):
        signal = Q @ signal
    panels.append(
        plu.Points(
            points,
            signal,
            size=2,
            title=f"$Q^{{{n_steps}}}\\,\\delta$   (mass = {np.sum(masses * signal):.4f})",
        )
    )

plu.plot_panels(panels, cmap="inferno")

## Spectral analysis

`Q` is self-adjoint for the mass-weighted inner product, so its eigenpairs come
from a symmetric Lanczos solver (`diffusion_eigsh`). The leading eigenvectors
behave like low-frequency Laplacian eigenfunctions:

In [ ]:
evals_Q, evecs = diffusion_eigsh(Q, k=40)

panels = [plu.Points(points, evecs[:, k], size=2, title=f"eigenvector {k}") for k in [1, 2, 3, 8]]
plu.plot_panels(panels, cmap="coolwarm")

## Comparison with a mesh Laplacian (qualitative)

Diffusion eigenvalues $\lambda^Q \in (0, 1]$ can be mapped to Laplacian-like
eigenvalues through the heat-kernel heuristic
$\lambda^\Delta = -\tfrac{2}{\sigma^2} \log \lambda^Q$. The resulting spectrum
*qualitatively* follows the FEM cotangent reference computed on the full mesh —
the correspondence is a modeling heuristic, not a numerical guarantee.

In [ ]:
L = skmesh.cotangent_laplacian(vertices, faces)
M = skmesh.fem_mass_matrix(vertices, faces)
evals_fem, _ = skmesh.fem_spectrum(L, M, k=40)

evals_pc = laplacian_eigenvalues(evals_Q, sigma)

plt.figure(figsize=(5.5, 4))
plt.plot(evals_fem, "k.-", label="FEM cotangent (mesh)")
plt.plot(evals_pc, "C1.-", label=r"$-\frac{2}{\sigma^2}\log \lambda^Q$ (point cloud)")
plt.xlabel("eigenvalue index")
plt.ylabel("eigenvalue")
plt.legend()
plt.grid(alpha=0.3)
plt.title("Spectra (qualitative comparison)")
plt.tight_layout()

## Where to go next

- `02_modalities_spectra.ipynb`: the same shape as mesh / point cloud /
  voxel grid / Gaussian mixture, all through the same normalization.
- `03_qdiffnet_layers.ipynb`: the learned (Q-DiffNet) side in PyTorch.